In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import math

# Primer

## What is WUENIC?
- Since 2001, WHO and UNICEF publish coverage estimates at country level
- Triangulation
    - Data sources
        - Administrative reports
        - "Official government estimates"
        - Survey evidence
    - Calculation
      - Add contextual information
      - Derive most likely figure
      - No confidence interval around the numbers but rather a "Grade of Confidence" figure

## The Data
- Administrative reports (Joint Reporting Form)
    - Who produces it?  Health facilities (monthly tally sheets) $\rightarrow$ District health offices (compile and transmit via DHIS2, District Vaccine Data Management Tool, or DVD-MT, or Excel) $\rightarrow$ MoH EPI (aggregates to national totals through JRF each Mar-May for previous calendar year) $\rightarrow$ WHO/UNICEF (validate internal consistency, e.g. $\ngtr$100%, logical age ranges)
    - Custodian?  WHO & UNICEF
    - Quantitative/Qualitative?  Quantitative
    - Frequency?  Annual
    - Components?  Numerator (surviving infants, pregnant women), Denominator (reported target population), result: "Administrative Coverage" =$\dfrac{\text{doses given}}{\text{target population}} \times 10$
    - Known limitations/biases:
        - Denominator errors (especially census-outdated populations)
        - Over-reporting (missed outreach or private sector)
        - Rarely adjusted for age-eligibility or multiple doses
        - Can exceed 100%
    - Typical Use in WUENIC?  Baseline trend, but is weighed cautiously due to data system limitations; subject to validation
- Official government estimates
    - Who produces it?  MoH/EPI/NITAG (National Immunization Technical Advisory Group)
    - Custodian?  WHO/UNICEF country offices
    - Quantitative/Qualitative?  Quantitative
    - Frequency?  Annual (Optional)
    - Components?  MoH reviews admin data (above) then issues an "official estimate"
    - Typical Use in WUENIC?  Used if endorsed and evidence-based
- Survey (DHS, MICS, NICS, etc.)
    - Who produces it?  National Statistics Office/MoH
    - Custodian?  WHO/UNICEF
    - Quantitative/Qualitative? Quantitative
    - Frequency?  $\approx$ 5-yearly
    - Typical Use in WUENIC?  Anchor for adjusting time trends
- Contextual information
    - Who produces it?  WHO/UNICEF field teams, partners
    - Custodian?  WHO/UNICEF
    - Quantitative/Qualitative?  Qualitative & quantitative
    - Frequency?  Continous
    - Typical Use in WUENIC?  Explains anomalies, guides confidence grade

In [2]:
path = "wuenic_input_to_pdf.xlsx"
df = pd.read_excel(path, sheet_name="wuenic_master")

print(df.shape)
print(df.dtypes)
df.head()                

(49683, 14)
Country                    object
ISOCountryCode             object
Vaccine                    object
Year                        int64
WUENIC                      int64
WUENICPreviousRevision    float64
GradeOfConfidence           int64
AdministrativeCoverage    float64
GovernmentEstimate        float64
ChildrenVaccinated        float64
ChildrenInTarget          float64
BirthsUNPD                  int64
SurvivingInfantsUNPD        int64
Comment                    object
dtype: object


,Country,ISOCountryCode,Vaccine,Year,WUENIC,WUENICPreviousRevision,GradeOfConfidence,AdministrativeCoverage,GovernmentEstimate,ChildrenVaccinated,ChildrenInTarget,BirthsUNPD,SurvivingInfantsUNPD,Comment
0,Afghanistan,AFG,BCG,1997,43,43.0,2,43.0,NaN,NaN,NaN,961674,896754,Legacy estimate. GoC=R+
1,Afghanistan,AFG,BCG,1998,35,35.0,2,NaN,35.0,NaN,NaN,985909,920265,Estimate informed by reported data. EPI Covera...
2,Afghanistan,AFG,BCG,1999,38,38.0,2,43.0,38.0,381008.0,878443.0,1013252,947416,Estimate informed by reported data. Afghanista...
3,Afghanistan,AFG,BCG,2000,30,30.0,2,38.0,30.0,412754.0,1079435.0,1036525,970393,Estimate informed by reported data. Data refle...
4,Afghanistan,AFG,BCG,2001,43,43.0,2,66.0,43.0,494628.0,747690.0,1006562,943488,Estimate informed by reported data. GoC=R+ D+


# Step 1: Pandas (Wrangling the data into shape)

Pandas is the workhorse.  Every other step depends on getting the data into a clean DataFrame, so we will spend real time here.  The five things every data scientist need to do with a new dataset: inspect it, clean it, reshape it, group it, and check it makes sense.

## Step 1.1: Inspect

In [7]:
df.info() # column types and non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49683 entries, 0 to 49682
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Country                 49683 non-null  object 
 1   ISOCountryCode          49683 non-null  object 
 2   Vaccine                 49683 non-null  object 
 3   Year                    49683 non-null  int64  
 4   WUENIC                  49683 non-null  int64  
 5   WUENICPreviousRevision  47230 non-null  float64
 6   GradeOfConfidence       49683 non-null  int64  
 7   AdministrativeCoverage  40267 non-null  float64
 8   GovernmentEstimate      41478 non-null  float64
 9   ChildrenVaccinated      37694 non-null  float64
 10  ChildrenInTarget        37588 non-null  float64
 11  BirthsUNPD              49683 non-null  int64  
 12  SurvivingInfantsUNPD    49683 non-null  int64  
 13  Comment                 49683 non-null  object 
dtypes: float64(5), int64(5), object(4)
mem

In [8]:
df.describe() # numeric summary

,Year,WUENIC,WUENICPreviousRevision,GradeOfConfidence,AdministrativeCoverage,GovernmentEstimate,ChildrenVaccinated,ChildrenInTarget,BirthsUNPD,SurvivingInfantsUNPD
count,49683.00000,49683.000000,47230.000000,49683.000000,40267.000000,41478.000000,3.769400e+04,3.758800e+04,4.968300e+04,4.968300e+04
mean,2012.36117,85.767164,85.835804,1.573959,87.489582,87.179951,6.348776e+05,2.599648e+08,6.918657e+05,6.722877e+05
std,7.88495,16.948835,17.095185,0.635570,16.930208,15.801441,2.115503e+06,5.025936e+10,2.254621e+06,2.187064e+06
min,1997.00000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000e+00,3.000000e+00,2.500000e+01,2.500000e+01
25%,2006.00000,81.000000,81.000000,1.000000,83.000000,83.000000,2.261900e+04,2.823700e+04,2.831900e+04,2.816200e+04
50%,2013.00000,92.000000,93.000000,1.000000,93.000000,93.000000,1.266350e+05,1.536770e+05,1.320070e+05,1.296980e+05
75%,2019.00000,97.000000,97.000000,2.000000,98.000000,97.000000,5.097890e+05,6.142830e+05,5.816560e+05,5.688310e+05
max,2024.00000,99.000000,99.000000,3.000000,890.000000,169.000000,3.598402e+07,9.744096e+12,2.932493e+07,2.787432e+07


In [9]:
df.isna().sum() # missing values per column

Country                       0
ISOCountryCode                0
Vaccine                       0
Year                          0
WUENIC                        0
WUENICPreviousRevision     2453
GradeOfConfidence             0
AdministrativeCoverage     9416
GovernmentEstimate         8205
ChildrenVaccinated        11989
ChildrenInTarget          12095
BirthsUNPD                    0
SurvivingInfantsUNPD          0
Comment                       0
dtype: int64

In [10]:
df["Vaccine"].value_counts()

Vaccine
DTP1     5400
DTP3     5400
MCV1     5400
Pol3     5400
HepB3    4456
BCG      4389
Hib3     3904
RCV1     3740
MCV2     3475
PCV3     1880
HepBB    1876
IPV1     1823
RotaC    1278
YFV       854
IPV2      314
MENGA      94
Name: count, dtype: int64

In [11]:
df["Year"].agg(["min", "max"])

min    1997
max    2024
Name: Year, dtype: int64